# 랭체인(LangChain) 상품추천GPT 만들기(RecommendationGPT)
## 작성자 : AISchool ( http://aischool.ai/%ec%98%a8%eb%9d%bc%ec%9d%b8-%ea%b0%95%ec%9d%98-%ec%b9%b4%ed%85%8c%ea%b3%a0%eb%a6%ac/ )
## 속성기반 감정분석 데이터 다운받기 : https://www.aihub.or.kr/aihubdata/data/view.do?currMenu=115&topMenu=100&aihubDataSe=data&dataSetSn=71603

# LangChain 라이브러리 설치

In [ ]:
!pip install --upgrade --quiet langchain langchain-community langchain-openai langchain-classic
!pip install --upgrade --quiet openai tiktoken
!pip install --upgrade --quiet chromadb sentence-transformers pysqlite3-binary
!pip install --upgrade --quiet pypdf unstructured jq
!pip install --upgrade --quiet lark


# 상품추천GPT(RecommendationGPT) 만들기

# 상품 리뷰 데이터 다운로드 & 업로드하기

In [ ]:
!unzip download.tar

In [ ]:
# 02.라벨링데이터/SNS/04. IT기기

# IT기기 리뷰 데이터 읽어오기

In [ ]:
from langchain_community.document_loaders import DirectoryLoader
from langchain_community.document_loaders import JSONLoader

loader = DirectoryLoader('/content/Sample/02.라벨링데이터/SNS/04. IT기기', glob="**/*.json", loader_cls=JSONLoader, loader_kwargs = {'jq_schema':'.[]', 'text_content':False})
docs = loader.load()

In [ ]:
docs[0]

In [ ]:
len(docs)

# 한글 인코딩 처리

In [ ]:
def escape_quotes_in_raw_text(json_str):
    # "RawText" 키의 시작 인덱스 찾기
    start_index = json_str.find('"RawText": "') + len('"RawText": "')
    # 시작 인덱스로부터 "RawText" 값의 종료 인덱스 찾기
    end_index = json_str.find('", "', start_index)
    # "RawText" 값 내의 모든 쌍따옴표를 이스케이프 처리된 형태로 변경
    raw_text_value = json_str[start_index:end_index].replace('"', '\\"')
    # 변경된 "RawText" 값을 원본 문자열에 다시 삽입
    fixed_json_string = json_str[:start_index] + raw_text_value + json_str[end_index:]

    return fixed_json_string

In [ ]:
def escape_quotes_in_sentiment_text(json_str):
    # "SentimentText": "를 찾는 것으로 시작
    search_str = '"SentimentText": "'
    start_index = 0

    while True:
        # "SentimentText": " 의 시작 위치를 찾기
        start_index = json_str.find(search_str, start_index)
        if start_index == -1:  # 더 이상 찾을 수 없으면 반복 종료
            break

        # 시작 위치 조정 (키의 길이를 더해 실제 값의 시작점으로 이동)
        start_index += len(search_str)
        # 해당 값의 종료 위치 찾기 (다음 큰따옴표 위치)
        end_index = json_str.find('", "', start_index)
        if end_index == -1:  # 예외 처리: 형식에 맞지 않는 경우
            break

        # SentimentText 값 내의 모든 큰따옴표 이스케이프 처리
        sentiment_text = json_str[start_index:end_index].replace('"', '\\"')
        # 변경된 값을 원본 문자열에 다시 삽입
        json_str = json_str[:start_index] + sentiment_text + json_str[end_index:]

        # 다음 SentimentText 검색을 위해 start_index 업데이트
        start_index = end_index + len('", "')

    return json_str

In [ ]:
import json

refined_docs = []

# 유니코드 이스케이프 시퀀스를 정상적인 문자열로 변환
for idx, doc in enumerate(docs):
    broken_korean = doc.page_content
    fixed_korean = broken_korean.encode('latin1').decode('unicode-escape')

    # "RawText" 부분에 "문자열을 \\"로 변경"
    fixed_json_string = escape_quotes_in_raw_text(fixed_korean)

    # "SentimentText" 부분에 "문자열을 \\"로 변경"
    fixed_json_string = escape_quotes_in_sentiment_text(fixed_json_string)

    refined_doc = json.loads(fixed_json_string)
    refined_docs.append(refined_doc)

    print(idx, fixed_json_string)

# Pandas Dataframe 형태로 정제하고 데이터 분석하기

In [ ]:
refined_docs[0]

In [ ]:
len(refined_docs)

In [ ]:
import pandas as pd

data_df = pd.DataFrame(refined_docs)
data_df

In [ ]:
unique_values_list = list(data_df['MainCategory'].unique())
print(unique_values_list)

In [ ]:
unique_values_list = list(data_df['ReviewScore'].unique())
print(unique_values_list)

In [ ]:
unique_values_list = list(data_df['RDate'].unique())
print(unique_values_list)

In [ ]:
unique_values_list = list(data_df['ProductName'].unique())
for idx, name in enumerate(unique_values_list):
    print(idx, name)

In [ ]:
data_df.columns

In [ ]:
data_df = data_df[['RawText', 'MainCategory', 'ProductName',
       'ReviewScore', 'RDate', 'GeneralPolarity',
       'Aspects']]
data_df

# GPT를 이용한 Attribute Info 추출

In [ ]:
OPENAI_KEY = "Input Your Key"

In [ ]:
data_df.head()

In [ ]:
from langchain_openai import ChatOpenAI

model = ChatOpenAI(model="gpt-4o-mini", openai_api_key=OPENAI_KEY)

prompt_text = (
    "Below is a table with information about IT products. " # 상품 추천 테마에 맞게 수정
    "Return a JSON list with an entry for each column. Each entry should have "
    '{"name": "column name", "description": "column description", "type": "column data type"}'
    f"\n\n{data_df.head()}\n\nJSON:\n"
)

response = model.invoke(prompt_text)
res = response.content

print(res)

In [ ]:
res

In [ ]:
import json
#마크다운 제거
clean_res = res.replace("```json", "").replace("```", "").strip()
attribute_info = json.loads(clean_res)
attribute_info

In [ ]:
# 변경 가능한 타입을 포함하는 열을 제외하고 nunique()를 다시 시도
safe_columns = [col for col in data_df.columns if not isinstance(data_df[col].iloc[0], list)]
unique_counts_safe = data_df[safe_columns].nunique()
unique_counts_safe

In [ ]:
attribute_info[-6][
    "description"
] += f". Valid values are {sorted(data_df['MainCategory'].value_counts().index.tolist())}"
attribute_info[-4][
    "description"
] += f". Valid values are {sorted(data_df['ReviewScore'].value_counts().index.tolist())}"
attribute_info[-2][
    "description"
] += f". Valid values are {sorted(data_df['GeneralPolarity'].value_counts().index.tolist())}"

In [ ]:
attribute_info

# Creating a query constructor chain


In [ ]:
from langchain_classic.chains.query_constructor.base import (
    get_query_constructor_prompt,
    load_query_constructor_runnable,
)

In [ ]:
doc_contents = "리뷰에 대한 자세한 설명" #Dummy Text
prompt = get_query_constructor_prompt(doc_contents, attribute_info)
print(prompt.format(query="{query}"))

In [ ]:
chain = load_query_constructor_runnable(
    ChatOpenAI(model="gpt-4o-mini", temperature=0, openai_api_key=OPENAI_KEY), doc_contents, attribute_info
)

In [ ]:
chain.invoke({"query": "평점 5점인 카메라를 추천해줘"})

In [ ]:
chain.invoke({"query": "평점 5점인 무선키보드를 추천해줘"})

In [ ]:
chain.invoke({"query": "조용한 무선키보드를 추천해줘"}) #매칭이 제대로 안됨

In [ ]:
chain.invoke({"query": "조용한 무선키보드를 추천해줘. 쿼리는 무조건 한국어만 사용해"})

# Adding examples specific to our use case


**'조용한 무선 키보드를 추천해줘'라는 요청 query가 영어로 변경된 모습**을 볼 수 있습니다.
따라서 **query 마지막에 ProductName: {키워드}를 추가한 few-shot 예시를 통해 query가 영어로 변하지 않고, ProductName을 제대로 매칭하도록 강제**합니다. 사용 사례별 예제를 추가하는 것이 도움이 될지 봅시다:

In [ ]:
examples = [
    (
        "조용한 무선 키보드를 추천해줘.",
        {
            "query": "조용한 무선 키보드 ProductName:키보드",
            "filter": '',
        },
    ),
    (
        "카메라를 추천해줘",
        {
            "query": "카메라 ProductName:카메라",
            "filter": '',
        },
    ),
    (
        "평점이 5점이 카메라를 추천해줘",
        {
            "query": "카메라 ProductName:카메라",
            "filter": 'eq("ReviewScore", "5")',
        },
    ),
]
prompt = get_query_constructor_prompt(
    doc_contents, attribute_info, examples=examples
)
print(prompt.format(query="{query}"))

In [ ]:
chain = load_query_constructor_runnable(
    ChatOpenAI(model="gpt-4o-mini", temperature=0, openai_api_key=OPENAI_KEY),
    doc_contents,
    attribute_info,
    examples=examples,
)

In [ ]:
chain.invoke({"query": "조용한 무선키보드를 추천해줘"})

In [ ]:
chain.invoke(
    {
        "query": "평점 4점인 스피커를 추천해줘"
    }
)

# Using with a self-querying retriever


이제 우리의 쿼리 구성 체인이 괜찮은 상태에 있다고 판단되므로, 실제 retriever와 함께 사용해 보겠습니다.

In [ ]:
from langchain_community.vectorstores import Chroma
from langchain_community.embeddings import HuggingFaceEmbeddings

model_name = "jhgan/ko-sroberta-multitask"
model_kwargs = {'device': 'cuda'} # Colab T4 GPU 활용
encode_kwargs = {'normalize_embeddings': False} # 모델 특성에 맞게 설정

embeddings = HuggingFaceEmbeddings(
    model_name=model_name,
    model_kwargs=model_kwargs,
    encode_kwargs=encode_kwargs
)

# Populating vectorstore


In [ ]:
from langchain_core.documents import Document

docs = []
for _, data in data_df.fillna("").iterrows():
    data_dict = data.to_dict()
    print(data_dict)
    # Aspects column은 제외
    aspect_exclude_data_dict = {k: v for k, v in data_dict.items() if k != 'Aspects'}

    doc = Document(
        page_content=json.dumps(data_dict, indent=2, ensure_ascii=False),
        metadata=aspect_exclude_data_dict
    )
    docs.append(doc)

In [ ]:
docs[0]

In [ ]:
docs

In [ ]:
len(docs)

In [ ]:
vecstore = Chroma.from_documents(docs, embeddings)

In [ ]:
from langchain.retrievers import SelfQueryRetriever

retriever = SelfQueryRetriever(
    query_constructor=chain, vectorstore=vecstore, verbose=True
)

In [ ]:
retriever

In [ ]:
results = retriever.get_relevant_documents(
    "카메라를 추천해줘"
)

for res in results:
    result_dict = json.loads(res.page_content)
    print(result_dict['ProductName'])
print('-----------------------------------------------------------------------------')

for res in results:
    print(res.page_content)
    print("\n" + "-" * 20 + "\n")

In [ ]:
results = retriever.get_relevant_documents(
    "평점 5점인 카메라를 추천해줘"
)
for res in results:
    result_dict = json.loads(res.page_content)
    print(result_dict['ProductName'])
print('-----------------------------------------------------------------------------')

for res in results:
    print(res.page_content)
    print("\n" + "-" * 20 + "\n")

In [ ]:
results = retriever.get_relevant_documents(
    "평점 5점인 무선키보드를 추천해줘"
)
for res in results:
    result_dict = json.loads(res.page_content)
    print(result_dict['ProductName'])
print('-----------------------------------------------------------------------------')

for res in results:
    print(res.page_content)
    print("\n" + "-" * 20 + "\n")

In [ ]:
results = retriever.get_relevant_documents(
    "조용한 무선키보드 제품을 추천해줘."
)
for res in results:
    result_dict = json.loads(res.page_content)
    print(result_dict['ProductName'])
print('-----------------------------------------------------------------------------')

for res in results:
    print(res.page_content)
    print("\n" + "-" * 20 + "\n")

In [ ]:
results = retriever.get_relevant_documents(
    "평점 5점인 음질이 좋은 스피커를 추천해줘"
)
for res in results:
    result_dict = json.loads(res.page_content)
    print(result_dict['ProductName'])
print('-----------------------------------------------------------------------------')

for res in results:
    print(res.page_content)
    print("\n" + "-" * 20 + "\n")

In [ ]:
results = retriever.get_relevant_documents(
    "평점 5점인 가벼운 노트북을 추천해줘"
)
for res in results:
    result_dict = json.loads(res.page_content)
    print(result_dict['ProductName'])
print('-----------------------------------------------------------------------------')

for res in results:
    print(res.page_content)
    print("\n" + "-" * 20 + "\n")